# Latest eval results (2026-09-26)

The two use cases have separate evals. This notebook shows the latest runs of each; nothing here calls the API.

1. **Research briefs** (`evals/run_eval.py`, `evals/research_set.json`): the goal and reader-level items, re-run after the prompt and tooling fixes. Code graders only (brief format, freshness): the judged graders' packets are written but **not yet graded**. The two original research items were fully graded in the pilot, shown at the end of this section.
2. **Fact verification, ambiguity set** (`evals/run.py`, graded by hand in the Claude Code session): v7 (shorter answer line) against the v4 baseline, plus targeted re-runs of the fixes that followed.

For the full 250-item fact-verification suite (v3), see `evals/results.ipynb`; for the history and reasoning behind every change, `evals/SUMMARY.md`.

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

RUNS = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "evals" / "runs").is_dir()) / "evals" / "runs"
pd.set_option("display.max_colwidth", 90)


def jsonl(path):
    return [json.loads(l) for l in open(path) if l.strip()]

## 1. Research briefs

| Run | Items | Code state |
|---|---|---|
| `20260926-150450` | 6 goal items | Less strict claim check, lighter goal-section prompt, first table rule (which over-triggered) |
| `20260926-152603` | 8 reader-level items (4 beginner/expert pairs) | Plus the table fix: default no table, `table.purpose` required, send-back check |

Per item: prose words (overview + sections; recommended 300-500, fails above 625), claim-check warnings left after the one revision round, whether a table was used, the two code grades, cost and time.

In [2]:
def research_table(run):
    rows = []
    for r in jsonl(RUNS / run / "results.jsonl"):
        s, g = r.get("structured") or {}, r.get("grades", {})
        prose = sum(len(str(p.get("statement", "")).split()) for p in s.get("overview", []) + [q for sec in s.get("sections", []) for q in sec.get("points", [])])
        rows.append({
            "item": r["id"], "question": r["question"][:70],
            "prose words": prose, "claim warnings": len(r["warnings"]),
            "table": "yes" if (s.get("table") or {}).get("rows") else "",
            "brief_format": g.get("brief_format", {}).get("score"),
            "format failures": "; ".join(g.get("brief_format", {}).get("failures", [])),
            "freshness": g.get("freshness", {}).get("score"),
            "$": r["cost_usd"], "s": round(r["elapsed_s"]),
        })
    return pd.DataFrame(rows)

goal = research_table("20260926-150450_claude-sonnet-4-6")
level = research_table("20260926-152603_claude-sonnet-4-6")
display(Markdown("### Goal items")); display(goal)
display(Markdown("### Reader-level pairs")); display(level)

### Goal items

,item,question,prose words,claim warnings,table,brief_format,format failures,freshness,$,s
0,goal-01,I'm visiting Kyoto next spring for the first time. Give me a short his,577,0,yes,1.000,,NaN,0.2595,138
1,goal-02,I want to switch careers into data analysis and people tell me to lear,511,3,yes,0.857,no citation warnings,1.0,0.1858,111
2,goal-03,I'm writing a high-school report on the causes of the French Revolutio,582,0,yes,0.857,further-reading notes are short (1 over 20 words),NaN,0.2157,132
3,goal-04,We're thinking about buying our first electric car. Explain how EV bat,585,5,yes,0.857,no citation warnings,1.0,0.3151,154
4,goal-05,I'm preparing for a school debate arguing against nuclear power. Give,566,3,yes,0.857,no citation warnings,NaN,0.3707,200
5,goal-06,I just adopted a retired racing greyhound. Tell me about the breed's h,453,0,yes,1.000,,NaN,0.1425,85


### Reader-level pairs

,item,question,prose words,claim warnings,table,brief_format,format failures,freshness,$,s
0,level-01a,My kid asked me how mRNA vaccines work and I had no idea what to say.,359,0,,1.000,,NaN,0.0955,38
1,level-01b,I'm an immunology PhD student. Give me an overview of mRNA vaccine pla,615,4,yes,0.714,further-reading notes are short (2 over 20 words); no citation warnings,NaN,0.4661,298
2,level-02a,I keep hearing about quantum entanglement in sci-fi shows. What is it,417,2,,0.714,further-reading notes are short (1 over 20 words); no citation warnings,NaN,0.1825,90
3,level-02b,I've taken graduate quantum mechanics. Summarize the history of Bell t,610,5,yes,0.857,no citation warnings,NaN,0.3964,252
4,level-03a,Why does everything cost more than it used to? Explain inflation like,592,0,,0.857,further-reading notes are short (3 over 20 words),NaN,0.2030,94
5,level-03b,I'm an economics grad student. Give me an overview of the main schools,681,6,,0.571,prose not far over the recommended 500 words (681; fails above 625); further-reading n...,1.0,0.4185,261
6,level-04a,What is CRISPR and why is everyone so excited about it?,584,1,,0.857,no citation warnings,1.0,0.2432,106
7,level-04b,I work in a molecular biology lab. Compare CRISPR-Cas9 nuclease editin,557,5,yes,0.857,no citation warnings,1.0,0.4205,248


In [3]:
both = pd.concat([goal.assign(set="goal"), level.assign(set="reader-level")])
summary = both.groupby("set").agg(items=("item", "count"), median_words=("prose words", "median"),
                                  over_625=("prose words", lambda w: int((w > 625).sum())),
                                  warnings_per_brief=("claim warnings", "mean"), tables=("table", lambda t: int((t == "yes").sum())),
                                  brief_format=("brief_format", "mean"), cost=("$", "sum"), mean_s=("s", "mean")).round(2)
summary

,items,median_words,over_625,warnings_per_brief,tables,brief_format,cost,mean_s
set,,,,,,,,
goal,6,571.5,0,1.83,6,0.9,1.49,136.67
reader-level,8,588.0,1,2.88,3,0.8,2.43,173.38


**Reading the research results**

- **Depth follows the reader's level.** In every pair the expert brief is longer (557-681 vs 359-592 words), reads more sections and costs 2-4x more. Level fit itself is a judged grade, still pending.
- **The table fix works on these items.** Tables appear only in three expert briefs that compare methods or platforms. In the goal run, before the fix, all six goal briefs had one.
- **Expert briefs keep more unsupported claims** (4-6 claim-check warnings, against 0-2 for beginner briefs). Technical detail is harder to keep strictly within the cited text.
- **Goal sections are still advice-heavy** (e.g. "Focus first on pandas…"). The lighter-goal prompt didn't hold; a structural fix was proposed and deferred.

### Pilot: the two original research items, fully graded (run `20260925-231835`, before later fixes)

In [4]:
pilot = [r for r in jsonl(RUNS / "20260925-231835_claude-sonnet-4-6" / "results.jsonl") if r["id"].startswith("research-")]
pd.DataFrame([{"item": r["id"], **{k: (v.get("score") if isinstance(v, dict) else None) for k, v in r["grades"].items()}} for r in pilot])

,item,brief_format,citations,completeness,relevance,style
0,research-01,1.000,0.824,0.944,0.971,0.583
1,research-02,0.875,0.895,1.000,0.981,0.500


## 2. Fact verification: ambiguity set

**v7** changed one thing from the shipped system: the `answer` field's description now asks for the bare answer (about 10 words at most), with the reading or scope, and extra facts, moved to the first reasoning item. Graded by hand under the strict footnote rule, like v4.

In [5]:
def amb(run):
    resp = {r["id"]: r for r in jsonl(RUNS / run / "responses.jsonl")}
    judg = {j["id"]: j for j in jsonl(RUNS / run / "judgments.jsonl")}
    return resp, judg

(r4, j4), (r7, j7) = amb("suite_v4_fix13"), amb("suite_v7_short_answer")

def answer_words(r):
    s = (r.get("turn1") or {}).get("structured") or {}
    return len(s["answer"].split()) if isinstance(s.get("answer"), str) else None

rows = []
for run, resp, judg in (("v4 (baseline)", r4, j4), ("v7 (short answer)", r7, j7)):
    words = pd.Series([answer_words(r) for r in resp.values()]).dropna()
    verdicts = pd.Series([j["verdict"] for j in judg.values()])
    rows.append({"run": run, "pass": int((verdicts == "pass").sum()), "partial": int((verdicts == "partial").sum()),
                 "fail": int((verdicts == "fail").sum()), "asked": sum(1 for r in resp.values() if (r.get("turn1") or {}).get("clarification")),
                 "median answer words": words.median(), "answers over 12 words": f"{(words > 12).mean():.0%}"})
pd.DataFrame(rows)

,run,pass,partial,fail,asked,median answer words,answers over 12 words
0,v4 (baseline),48,0,2,9,17.0,60%
1,v7 (short answer),45,1,4,10,9.0,34%


In [6]:
changed = [{"item": i, "v4": j4[i]["verdict"], "v7": j7[i]["verdict"], "v7 failure mode": j7[i]["primary_failure_mode"],
            "v7 rationale": j7[i].get("rationale", "")[:160]}
           for i in sorted(j7) if j4.get(i, {}).get("verdict") != j7[i]["verdict"] or j7[i]["verdict"] != "pass"]
pd.DataFrame(changed)

,item,v4,v7,v7 failure mode,v7 rationale
0,amb-01t,pass,fail,missed_ambiguity,"San Siro only (75,817); Inter Miami never mentioned. v4 covered both. Possible effect ..."
1,amb-09t,fail,fail,missed_ambiguity,Research brief on pulmonary embolism only; pre-eclampsia never mentioned (same failure...
2,amb-11t,pass,fail,missed_ambiguity,Programming language only; the board game never mentioned. The disambiguation check re...
3,amb-18c,pass,fail,unnecessary_clarification,"Asked, although late March rules out the NFL (v4 answered Florida Panthers). The agent..."
4,amb-22t,fail,pass,none,Asked golf / snooker / darts / curling (v4 failed this item). Follow-up: Rory McIlroy.
5,amb-23t,pass,partial,missed_ambiguity,"Answered rugby league (Hull KR) and only mentioned that other Super Leagues exist, wit..."


**Reading the ambiguity results**

- **Answer lines roughly halved** (median 17 to 9 words); pass rate 45/50 + 1 partial vs v4's 48/50.
- **Only amb-01t (Inter) plausibly traces to the shorter answer**: it gave one reading and never named the other. amb-11t (Go) and amb-18c (Panthers) trace to the question-only disambiguation check and to search-path variance; amb-22t (Masters) improved.
- **Run-to-run variance is large and unmeasured.** Items flipped between runs with identical code (amb-03t, amb-11t). No configuration has been run with repeats, so a 3-item swing on 50 cannot be told apart from noise.

### Follow-up fixes after v7 (targeted re-runs, not fully graded)

| Run | What changed | Items | Outcome |
|---|---|---|---|
| `suite_v7b_disamb_fix` | The disambiguation check gets the whole page as a compact list (the old 6,000-character cut hid "Go (programming language)", and hid entries on Mercury, Springfield, Panther and Master) | 6 long-page items | 6/6 pass, including amb-18c (was fail). Run directly, the check still resolves amb-11t's "Go" to the language: a calibration issue, not truncation |
| `suite_v7c_readings` | Readings rule: short answer line, one reasoning item per reading | 14 multi-reading items | Answer lines shorter on 7 of 12 answered items but still 13-28 words; exposed a truncation bug (amb-03t) |
| `suite_v7d_complete` | A submission cut off by the output limit, or missing reasoning or sources, is sent back instead of accepted | amb-03t | Complete answer, with reasoning and sources; this time it covered 2 of the 3 Rangers teams |